# Train DistilRoBERTa on CUAD (#43)

Run this in Google Colab with a **GPU runtime**:
Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

The run uses the fixed config in `configs/transformer.yaml`. Nothing is tuned, and
the test set is used once, at the end. There is no validation split, by the
advisor's decision.

Expect roughly 30-60 minutes on a T4.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the repository

The repo is private, so you need a GitHub personal access token:
GitHub -> Settings -> Developer settings -> Personal access tokens -> Tokens (classic),
with the `repo` scope.

Paste it when prompted. It is not saved anywhere.

In [ ]:
import getpass

BRANCH = "43-train-the-distilroberta-multi-label-classifier"
token = getpass.getpass("GitHub token: ")

!git clone --branch {BRANCH} --depth 1 https://{token}@github.com/Break-Through-Tech/Accenture-1Q-contract-review-challenge.git repo
%cd repo
!git log --oneline -1

## 3. Install dependencies

Colab already has torch, pandas and pyarrow. Only `transformers` usually needs installing.

In [ ]:
!pip install -q transformers pyyaml

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 4. Build the training data

The parquet files are gitignored, so they are rebuilt here from the committed raw
CUAD JSON. This cleans each contract, chunks the cleaned text, then maps spans to
chunk labels. Takes about a minute.

In [ ]:
!python -m src.ingest.build_dataset

## 5. Train and evaluate

Trains on the 408 official train contracts and evaluates once on the 102 test
contracts. Chunk predictions are pooled to contract level (max probability per
category) before scoring, so a clause sitting in a chunk overlap is not counted twice.

In [ ]:
import time

from src.models.trainer import train_and_evaluate

start = time.time()
result = train_and_evaluate("configs/transformer.yaml", device="cuda")
print(f"\ntotal time: {(time.time() - start) / 60:.1f} min")

## 6. Results

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 140)

metrics = result["per_category_metrics"].sort_values("f1", ascending=False)
print("SUMMARY")
for key, value in result["summary"].items():
    print(f"  {key}: {value}")
print()
metrics

In [ ]:
# Sanity check against the naive baseline from the EDA: 67.4% accuracy, 0% recall.
# A model that predicts nothing scores macro-F1 = 0, so anything above 0 beats it.
summary = result["summary"]
print(f"macro-F1: {summary['macro_f1']:.4f}")
print(f"categories with F1 > 0: {(metrics['f1'] > 0).sum()} of {len(metrics)}")
print(f"categories never predicted: {(metrics['predicted_positive'] == 0).sum()}")
print()
print("weakest 10 by F1 (support = positive contracts in test):")
print(metrics.nsmallest(10, "f1")[["category", "precision", "recall", "f1", "support"]].to_string(index=False))

## 7. Save the run

The checkpoint is about 330 MB, so it goes to Drive rather than into git. The metrics
and predictions are small and are downloaded directly.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/cuad-runs"
!cp -r models/distilroberta-cuad "/content/drive/MyDrive/cuad-runs/"
!ls -la "/content/drive/MyDrive/cuad-runs/distilroberta-cuad"

In [ ]:
from google.colab import files

# Small artifacts worth committing to the repo or attaching to the issue.
files.download("models/distilroberta-cuad/test_category_metrics.csv")
files.download("models/distilroberta-cuad/run_metadata.json")
files.download("models/distilroberta-cuad/training_history.csv")

## Notes for the write-up

`run_metadata.json` records the model, the resolved category list, the seed, the
threshold, and the full training config, so the run can be repeated.

Things to report alongside the numbers:

- **34 of 41 categories are trained.** The other 7 have no positive examples in train,
  so the model cannot predict them and they score 0 on test. Say whether macro-F1 is
  over 34 or 41 categories.
- **No threshold tuning.** The fixed 0.5 threshold is likely too high, since only 1.79%
  of chunk labels are positive. Tuning it is #7.
- **No validation split**, by the advisor's decision, so nothing was selected on
  held-out data.